# Import Modules

In [1]:
# Import Modules
import os
import zipfile
import urllib.request
import requests
import pandas as pd
import arcpy

from arcgis.gis import GIS
from arcgis.raster import ImageryLayer

from config import (
    CENSUS_API_KEY,
    PROJECT_FOLDER,
    CITY_NAME,
    STATE_FIPS,
    COUNTY_FIPS,
    ACS_YEAR,
    LANDSAT_DATE_RANGE,
    MAX_CLOUD_COVER,
    NLCD_TCC_YEAR
)

# Create Folders

In [4]:
raw_folder = os.path.join(PROJECT_FOLDER, "data", "raw")
processed_folder = os.path.join(PROJECT_FOLDER, "data", "processed")
outputs_folder = os.path.join(PROJECT_FOLDER, "outputs")
tables_folder = os.path.join(outputs_folder, "tables")
figures_folder = os.path.join(outputs_folder, "figures")
publish_folder = os.path.join(outputs_folder, "publish")

for folder in [raw_folder, processed_folder, outputs_folder, tables_folder, figures_folder, publish_folder]:
    os.makedirs(folder, exist_ok=True)


# Create GDB

In [5]:
gdb_path = os.path.join(PROJECT_FOLDER, "UrbanHeatEquity.gdb")

if not arcpy.Exists(gdb_path):
    arcpy.management.CreateFileGDB(PROJECT_FOLDER, "UrbanHeatEquity.gdb")

arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True

print("Project initialized.")

Project initialized.


# Download Arizona Places and select City of Phoenix boundary

In [6]:
place_url = f"https://www2.census.gov/geo/tiger/TIGER2024/PLACE/tl_2024_{STATE_FIPS}_place.zip"
place_zip = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_place.zip")
place_extract = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_place")

if not os.path.exists(place_zip):
    print("Downloading TIGER Places...")
    urllib.request.urlretrieve(place_url, place_zip)

if not os.path.exists(place_extract):
    os.makedirs(place_extract, exist_ok=True)
    with zipfile.ZipFile(place_zip, "r") as zip_ref:
        zip_ref.extractall(place_extract)

place_shp = [os.path.join(place_extract, f) for f in os.listdir(place_extract) if f.endswith(".shp")][0]

places_all = os.path.join(gdb_path, "az_places_all")
city_boundary = os.path.join(gdb_path, "city_of_phoenix_boundary")

for fc in [places_all, city_boundary]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

arcpy.conversion.FeatureClassToFeatureClass(place_shp, gdb_path, "az_places_all")
arcpy.analysis.Select(places_all, city_boundary, "NAME = 'Phoenix'")

print("City boundary created:", city_boundary)

City boundary created: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\UrbanHeatEquity.gdb\city_of_phoenix_boundary


# Get Phoenix extent for raster searches

In [7]:
desc = arcpy.Describe(city_boundary)
extent = desc.extent

BBOX = {
    "xmin": extent.XMin,
    "ymin": extent.YMin,
    "xmax": extent.XMax,
    "ymax": extent.YMax
}

print("Phoenix extent:", BBOX)

Phoenix extent: {'xmin': -112.32406500029947, 'ymin': 33.29025999969929, 'xmax': -111.92552099989422, 'ymax': 33.9183909998448}


# Search Landsat Collection 2 Level-2 imagery

In [8]:
landsat_folder = os.path.join(raw_folder, "landsat")
os.makedirs(landsat_folder, exist_ok=True)

stac_search_url = "https://planetarycomputer.microsoft.com/api/stac/v1/search"

search_payload = {
    "collections": ["landsat-c2-l2"],
    "bbox": [BBOX["xmin"], BBOX["ymin"], BBOX["xmax"], BBOX["ymax"]],
    "datetime": LANDSAT_DATE_RANGE,
    "query": {
        "eo:cloud_cover": {"lt": MAX_CLOUD_COVER},
        "platform": {"in": ["landsat-8", "landsat-9"]}
    },
    "limit": 10
}

response = requests.post(stac_search_url, json=search_payload)
response.raise_for_status()

features = response.json()["features"]

if len(features) == 0:
    raise RuntimeError("No Landsat scenes found. Try expanding the date range or cloud threshold.")

for i, feature in enumerate(features):
    print(i, feature["id"], feature["properties"].get("datetime"), feature["properties"].get("eo:cloud_cover"))

0 LC08_L2SP_036037_20230927_02_T1 2023-09-27T17:57:56.796638Z 0.0
1 LC09_L2SP_037037_20230926_02_T1 2023-09-26T18:04:01.048193Z 0.0
2 LC09_L2SP_037036_20230926_02_T1 2023-09-26T18:03:37.157178Z 0.0
3 LC09_L2SP_036037_20230919_02_T1 2023-09-19T17:57:54.457922Z 0.0
4 LC08_L2SP_037037_20230918_02_T1 2023-09-18T18:04:04.972731Z 0.0
5 LC08_L2SP_037036_20230918_02_T1 2023-09-18T18:03:41.081692Z 0.97
6 LC09_L2SP_036037_20230903_02_T1 2023-09-03T17:57:47.705021Z 4.99


# Select first unique path/row scenes

In [9]:
selected_scenes = []
seen_pathrows = set()

for feature in features:
    scene_id = feature["id"]
    pathrow = scene_id.split("_")[2]

    if pathrow not in seen_pathrows:
        selected_scenes.append(feature)
        seen_pathrows.add(pathrow)

    if len(selected_scenes) == 2:
        break

print("Selected Landsat scenes:")
for scene in selected_scenes:
    print(scene["id"])

Selected Landsat scenes:
LC08_L2SP_036037_20230927_02_T1
LC09_L2SP_037037_20230926_02_T1


# Download Landsat thermal and QA assets

In [10]:
def sign_pc_href(href):
    sign_url = "https://planetarycomputer.microsoft.com/api/sas/v1/sign"
    r = requests.get(sign_url, params={"href": href})
    r.raise_for_status()
    return r.json()["href"]

def download_scene_asset(scene, asset_key, output_name):
    if asset_key not in scene["assets"]:
        raise KeyError(f"{asset_key} not found. Available assets: {list(scene['assets'].keys())}")

    href = scene["assets"][asset_key]["href"]
    signed_href = sign_pc_href(href)
    out_path = os.path.join(landsat_folder, output_name)

    if not os.path.exists(out_path):
        print("Downloading:", output_name)
        urllib.request.urlretrieve(signed_href, out_path)
    else:
        print("Already exists:", output_name)

    return out_path

for scene in selected_scenes:
    scene_id = scene["id"]
    download_scene_asset(scene, "lwir11", f"{scene_id}_ST_B10.tif")
    download_scene_asset(scene, "qa_pixel", f"{scene_id}_QA_PIXEL.tif")

print("Landsat download complete.")

Already exists: LC08_L2SP_036037_20230927_02_T1_ST_B10.tif
Already exists: LC08_L2SP_036037_20230927_02_T1_QA_PIXEL.tif
Already exists: LC09_L2SP_037037_20230926_02_T1_ST_B10.tif
Already exists: LC09_L2SP_037037_20230926_02_T1_QA_PIXEL.tif
Landsat download complete.


# Search ArcGIS Online for NLCD Tree Canopy imagery

In [11]:
gis_public = GIS()

search_results = gis_public.content.search(
    query=f"NLCD TCC {NLCD_TCC_YEAR} Tree Canopy Cover",
    item_type="Imagery Layer",
    max_items=10
)

for i, item in enumerate(search_results):
    print(i, item.title, item.id)

0 National Land Cover Database (NLCD) 2023: Percent Tree Canopy Cover of New Jersey cd81ed788efa4a39bca82312cc749c74
1 Science Tree Canopy Cover (TCC) Hawaii 1524520acae743178f6328a2f0889bdb
2 Science Tree Canopy Cover (TCC) Conterminous United States 78592dac7150449d944c8fc838df221a
3 National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Southeast Alaska c6b442b00d5f4f6296a1fc2dc23f51e4
4 Science Tree Canopy Cover (TCC) Southeast Alaska b3418072ba7949c692696c8771acff88
5 National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Conterminous United States d44782ad936d436cad6ed91e7f49acb5
6 National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Puerto Rico USVI 50ad050c2cd44598a9bee2b4990de85e
7 Tree Canopy Cover (TCC) Science Standard Error (SE) Conterminous United States 5c654ea0ccc24b3bac6ddb2c3db3e469
8 Science Tree Canopy Cover (TCC) Puerto Rico USVI bd36fdde66794b29a414ec2a24b6a195
9 National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Hawaii 65e31c64b8f6467da4

# Select NLCD TCC imagery layer

In [12]:
tcc_item_index = 5  # Change this if result 0 is not the correct NLCD TCC layer
tcc_item = search_results[tcc_item_index]

print("Selected:", tcc_item.title)
print("URL:", tcc_item.url)

tcc_layer = ImageryLayer(tcc_item.url, gis=gis_public)

Selected: National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Conterminous United States
URL: https://imagery.geoplatform.gov/iipp/rest/services/Vegetation/USFS_EDW_NLCD_TCC_CONUS/ImageServer


# Export Phoenix-only tree canopy raster

In [13]:
tcc_output_folder = os.path.join(raw_folder, "nlcd_tree_canopy")
os.makedirs(tcc_output_folder, exist_ok=True)

tree_canopy_path = os.path.join(
    tcc_output_folder,
    f"nlcd_tcc_{NLCD_TCC_YEAR}_phoenix.tif"
)

bbox_string = f"{extent.XMin},{extent.YMin},{extent.XMax},{extent.YMax}"

if not os.path.exists(tree_canopy_path):
    print("Exporting Phoenix-only tree canopy raster...")
    exported = tcc_layer.export_image(
        bbox=bbox_string,
        bbox_sr=4326,
        image_sr=4326,
        size=[2500, 2500],
        export_format="tiff",
        f="image",
        save_folder=tcc_output_folder,
        save_file=os.path.basename(tree_canopy_path)
    )
    print("Exported:", exported)
else:
    print("Tree canopy raster already exists:", tree_canopy_path)

Tree canopy raster already exists: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\raw\nlcd_tree_canopy\nlcd_tcc_2023_phoenix.tif


# Download TIGER tracts and select whole tracts intersecting Phoenix

In [14]:
tiger_url = f"https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_{STATE_FIPS}_tract.zip"
tiger_zip = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_tract.zip")
tiger_extract = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_tract")

if not os.path.exists(tiger_zip):
    print("Downloading TIGER tracts...")
    urllib.request.urlretrieve(tiger_url, tiger_zip)

if not os.path.exists(tiger_extract):
    os.makedirs(tiger_extract, exist_ok=True)
    with zipfile.ZipFile(tiger_zip, "r") as zip_ref:
        zip_ref.extractall(tiger_extract)

tract_shp = [os.path.join(tiger_extract, f) for f in os.listdir(tiger_extract) if f.endswith(".shp")][0]

tracts_all = os.path.join(gdb_path, "state_tracts_all")
tracts_study = os.path.join(gdb_path, "phoenix_city_intersecting_tracts")

for fc in [tracts_all, tracts_study]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

arcpy.conversion.FeatureClassToFeatureClass(tract_shp, gdb_path, "state_tracts_all")

arcpy.management.MakeFeatureLayer(tracts_all, "tracts_lyr")
arcpy.management.SelectLayerByLocation(
    "tracts_lyr",
    "INTERSECT",
    city_boundary,
    selection_type="NEW_SELECTION"
)

arcpy.management.CopyFeatures("tracts_lyr", tracts_study)

count = int(arcpy.management.GetCount(tracts_study)[0])
print("Intersecting tracts selected:", count)

Intersecting tracts selected: 434


# Download ACS demographics from Census API

In [15]:
acs_vars = [
    "NAME",
    "B01003_001E",
    "B19013_001E",
    "B02001_002E",
    "B02001_003E",
    "B03003_003E",
    "B17001_002E",
    "B25077_001E",
    "B25064_001E",
    "B15003_022E"
]

base_url = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

params = {
    "get": ",".join(acs_vars),
    "for": "tract:*",
    "in": f"state:{STATE_FIPS} county:{COUNTY_FIPS}",
    "key": CENSUS_API_KEY
}

response = requests.get(base_url, params=params)

if response.headers.get("Content-Type", "").startswith("text/html"):
    print(response.text[:500])
    raise RuntimeError("Census API returned HTML instead of JSON. Check your API key.")

response.raise_for_status()
acs_json = response.json()

acs_df = pd.DataFrame(acs_json[1:], columns=acs_json[0])

acs_df["GEOID"] = acs_df["state"] + acs_df["county"] + acs_df["tract"]

acs_df = acs_df.rename(columns={
    "B01003_001E": "total_population",
    "B19013_001E": "median_income",
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B03003_003E": "hispanic_latino",
    "B17001_002E": "poverty_count",
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",
    "B15003_022E": "bachelors_degree"
})

numeric_cols = [
    "total_population",
    "median_income",
    "white_alone",
    "black_alone",
    "hispanic_latino",
    "poverty_count",
    "median_home_value",
    "median_gross_rent",
    "bachelors_degree"
]
for col in numeric_cols:
    acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")

acs_df["median_income"] = acs_df["median_income"].where(
    (acs_df["median_income"] > 0) &
    (acs_df["median_income"] < 300000)
)

acs_df["pct_black"] = acs_df["black_alone"] / acs_df["total_population"] * 100
acs_df["pct_hispanic"] = acs_df["hispanic_latino"] / acs_df["total_population"] * 100
acs_df["pct_poverty"] = acs_df["poverty_count"] / acs_df["total_population"] * 100
acs_df["pct_bachelors"] = acs_df["bachelors_degree"] / acs_df["total_population"] * 100

acs_cache = os.path.join(processed_folder, "acs_demographics.csv")
acs_df.to_csv(acs_cache, index=False)

print("ACS saved:", acs_cache)
acs_df.head()

ACS saved: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\processed\acs_demographics.csv


,NAME,total_population,median_income,white_alone,black_alone,hispanic_latino,poverty_count,median_home_value,median_gross_rent,bachelors_degree,state,county,tract,GEOID,pct_black,pct_hispanic,pct_poverty,pct_bachelors
0,Census Tract 101.02; Maricopa County; Arizona,6525,201667.0,5657,172,529,870,1751100,-666666666,1879,04,013,010102,04013010102,2.636015,8.107280,13.333333,28.796935
1,Census Tract 101.03; Maricopa County; Arizona,3842,143478.0,3336,27,407,440,936400,2732,797,04,013,010103,04013010103,0.702759,10.593441,11.452369,20.744404
2,Census Tract 101.04; Maricopa County; Arizona,3381,142356.0,3159,20,75,219,853700,2332,1398,04,013,010104,04013010104,0.591541,2.218279,6.477374,41.348713
3,Census Tract 304.01; Maricopa County; Arizona,4774,150943.0,4361,87,84,400,1107100,1538,1820,04,013,030401,04013030401,1.822371,1.759531,8.378718,38.123167
4,Census Tract 304.02; Maricopa County; Arizona,4237,106801.0,3862,0,206,278,988700,2493,1120,04,013,030402,04013030402,0.000000,4.861931,6.561246,26.433797


In [10]:
tcc_item_index = 5  # Change if a better result appears
tcc_item = search_results[tcc_item_index]

print("Selected:", tcc_item.title)
print("URL:", tcc_item.url)

tcc_layer = ImageryLayer(tcc_item.url, gis=gis)

Selected: National Land Cover Database (NLCD) Tree Canopy Cover (TCC) Conterminous United States
URL: https://imagery.geoplatform.gov/iipp/rest/services/Vegetation/USFS_EDW_NLCD_TCC_CONUS/ImageServer


# Export Phoenix-only tree canopy raster

In [11]:
tcc_output_folder = os.path.join(raw_folder, "nlcd_tree_canopy")
os.makedirs(tcc_output_folder, exist_ok=True)

tree_canopy_path = os.path.join(
    tcc_output_folder,
    f"nlcd_tcc_{NLCD_TCC_YEAR}_phoenix.tif"
)

bbox_string = f"{extent.XMin},{extent.YMin},{extent.XMax},{extent.YMax}"

exported = tcc_layer.export_image(
    bbox=bbox_string,
    bbox_sr=4326,
    image_sr=4326,
    size=[2500, 2500],
    export_format="tiff",
    f="image",
    save_folder=tcc_output_folder,
    save_file=os.path.basename(tree_canopy_path)
)

print(exported)

C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\raw\nlcd_tree_canopy\nlcd_tcc_2023_phoenix.tif


# Download TIGER tracts and select whole tracts intersecting Phoenix

In [12]:
tiger_url = f"https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_{STATE_FIPS}_tract.zip"
tiger_zip = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_tract.zip")
tiger_extract = os.path.join(raw_folder, f"tl_2024_{STATE_FIPS}_tract")

if not os.path.exists(tiger_zip):
    urllib.request.urlretrieve(tiger_url, tiger_zip)

if not os.path.exists(tiger_extract):
    os.makedirs(tiger_extract, exist_ok=True)
    with zipfile.ZipFile(tiger_zip, "r") as zip_ref:
        zip_ref.extractall(tiger_extract)

tract_shp = [os.path.join(tiger_extract, f) for f in os.listdir(tiger_extract) if f.endswith(".shp")][0]

tracts_all = os.path.join(gdb_path, "state_tracts_all")
tracts_study = os.path.join(gdb_path, "phoenix_city_intersecting_tracts")

for fc in [tracts_all, tracts_study]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

arcpy.conversion.FeatureClassToFeatureClass(tract_shp, gdb_path, "state_tracts_all")

arcpy.management.MakeFeatureLayer(tracts_all, "tracts_lyr")
arcpy.management.SelectLayerByLocation(
    "tracts_lyr",
    "INTERSECT",
    city_boundary,
    selection_type="NEW_SELECTION"
)

arcpy.management.CopyFeatures("tracts_lyr", tracts_study)

print(tracts_study)

C:\Users\ava_gotthard\Documents\UrbanHeatEquity\UrbanHeatEquity.gdb\phoenix_city_intersecting_tracts


# Download ACS demographics from Census API

In [13]:
acs_year = "2024"

acs_vars = [
    "NAME",
    "B01003_001E",
    "B19013_001E",
    "B02001_002E",
    "B02001_003E",
    "B03003_003E",
    "B17001_002E",
    "B25077_001E",
    "B25064_001E",
    "B15003_022E"
]

base_url = f"https://api.census.gov/data/{acs_year}/acs/acs5"

params = {
    "get": ",".join(acs_vars),
    "for": "tract:*",
    "in": f"state:{STATE_FIPS} county:{COUNTY_FIPS}",
    "key": CENSUS_API_KEY
}
response = requests.get(base_url, params=params)
response.raise_for_status()

acs_json = response.json()
acs_df = pd.DataFrame(acs_json[1:], columns=acs_json[0])

acs_df["GEOID"] = acs_df["state"] + acs_df["county"] + acs_df["tract"]

acs_df = acs_df.rename(columns={
    "B01003_001E": "total_population",
    "B19013_001E": "median_income",
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B03003_003E": "hispanic_latino",
    "B17001_002E": "poverty_count",
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",
    "B15003_022E": "bachelors_degree"
})

numeric_cols = [
    "total_population",
    "median_income",
    "white_alone",
    "black_alone",
    "hispanic_latino",
    "poverty_count",
    "median_home_value",
    "median_gross_rent",
    "bachelors_degree"
]

for col in numeric_cols:
    acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")

# Clean Census missing/suppressed income values
acs_df["median_income"] = acs_df["median_income"].where(
    (acs_df["median_income"] > 0) &
    (acs_df["median_income"] < 300000)
)

acs_df["pct_black"] = acs_df["black_alone"] / acs_df["total_population"] * 100
acs_df["pct_hispanic"] = acs_df["hispanic_latino"] / acs_df["total_population"] * 100
acs_df["pct_poverty"] = acs_df["poverty_count"] / acs_df["total_population"] * 100
acs_df["pct_bachelors"] = acs_df["bachelors_degree"] / acs_df["total_population"] * 100

acs_cache = os.path.join(processed_folder, "acs_demographics.csv")
acs_df.to_csv(acs_cache, index=False)

acs_df.head()

,NAME,total_population,median_income,white_alone,black_alone,hispanic_latino,poverty_count,median_home_value,median_gross_rent,bachelors_degree,state,county,tract,GEOID,pct_black,pct_hispanic,pct_poverty,pct_bachelors
0,Census Tract 101.02; Maricopa County; Arizona,6525,201667.0,5657,172,529,870,1751100,-666666666,1879,04,013,010102,04013010102,2.636015,8.107280,13.333333,28.796935
1,Census Tract 101.03; Maricopa County; Arizona,3842,143478.0,3336,27,407,440,936400,2732,797,04,013,010103,04013010103,0.702759,10.593441,11.452369,20.744404
2,Census Tract 101.04; Maricopa County; Arizona,3381,142356.0,3159,20,75,219,853700,2332,1398,04,013,010104,04013010104,0.591541,2.218279,6.477374,41.348713
3,Census Tract 304.01; Maricopa County; Arizona,4774,150943.0,4361,87,84,400,1107100,1538,1820,04,013,030401,04013030401,1.822371,1.759531,8.378718,38.123167
4,Census Tract 304.02; Maricopa County; Arizona,4237,106801.0,3862,0,206,278,988700,2493,1120,04,013,030402,04013030402,0.000000,4.861931,6.561246,26.433797
